User-based collaborative filtering recommends items by finding users with similar tastes to the target user and then suggesting items liked by those similar users.

In [3]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
df = pd.read_csv("feedback.csv")

In [5]:
df.head(1)

,user_id,content_id,like_dislike,rating,feedback_date
0,1,1193,1,5,2020-01-23 22:58:09


In [7]:
df = df[['user_id' , 'content_id', 'rating']]

In [9]:
df.isnull().sum()

user_id       0
content_id    0
rating        0
dtype: int64

In [11]:
# Create User x Movie ratings matrix
ratings_matrix = df.pivot_table(
    index='user_id',
    columns='content_id',
    values='rating'
)


In [17]:
ratings_matrix.head()

content_id,1,2,3,4,5,6,7,8,9,10,...,3943,3944,3945,3946,3947,3948,3949,3950,3951,3952
user_id,,,,,,,,,,,,,,,,,,,,,
1,5.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [15]:
ratings_matrix = ratings_matrix.fillna(0)   # Only for similarity calculation

In [39]:
# Compute cosine similarity
user_similarity = cosine_similarity(ratings_matrix)

# Put into a DataFrame for easy reading
user_similarity_df = pd.DataFrame(
    user_similarity, 
    index=ratings_matrix.index, 
    columns=ratings_matrix.index
)

In [41]:
user_similarity_df.head()

user_id,1,2,3,4,5,6,7,8,9,10,...,6031,6032,6033,6034,6035,6036,6037,6038,6039,6040
user_id,,,,,,,,,,,,,,,,,,,,,
1,1.000000,0.104503,0.080240,0.139364,0.088323,0.176237,0.064551,0.159835,0.221336,0.233912,...,0.146556,0.080653,0.078037,0.037355,0.119372,0.174214,0.148710,0.000000,0.155652,0.106137
2,0.104503,1.000000,0.136351,0.178245,0.098719,0.082870,0.290659,0.193232,0.190749,0.179796,...,0.102178,0.076037,0.255792,0.000000,0.164121,0.192397,0.191459,0.056510,0.070636,0.138937
3,0.080240,0.136351,1.000000,0.138626,0.041342,0.042642,0.095273,0.057144,0.118453,0.168866,...,0.064910,0.123463,0.144185,0.000000,0.078459,0.128389,0.079817,0.136864,0.091047,0.085269
4,0.139364,0.178245,0.138626,1.000000,0.037579,0.000000,0.111018,0.084471,0.086817,0.094529,...,0.143242,0.064226,0.346296,0.000000,0.057438,0.187024,0.132436,0.031271,0.051513,0.075842
5,0.088323,0.098719,0.041342,0.037579,1.000000,0.028513,0.101302,0.201471,0.211670,0.078053,...,0.060584,0.035464,0.032711,0.033996,0.125063,0.201885,0.118942,0.012002,0.034358,0.229942


In [51]:
target_user = 1000
similar_users = user_similarity_df[target_user].sort_values(ascending=False).drop(target_user).head(5)
print(similar_users)

user_id
2539    0.482567
5842    0.479641
3906    0.471494
1349    0.465062
1188    0.463439
Name: 1000, dtype: float64


In [53]:
# Movies already rated by target user
rated_by_target = ratings_matrix.loc[target_user]
rated_by_target = rated_by_target[rated_by_target > 0].index

# Get ratings of neighbors
neighbors = similar_users.index
neighbor_ratings = ratings_matrix.loc[neighbors]

# Weighted average of neighbor ratings
weighted_scores = (neighbor_ratings.T.dot(similar_users)) / similar_users.sum()

# Remove movies already rated by target user
recommendations = weighted_scores.drop(rated_by_target, errors='ignore')

# Top 10 recommended movies
top_recommendations = recommendations.sort_values(ascending=False).head(10)
print(top_recommendations)


content_id
1196    3.805815
2944    3.617891
1953    1.999624
2028    1.982378
3753    1.965328
1222    1.832389
3654    1.819831
3104    1.813638
3703    1.801530
474     1.769139
dtype: float64


In [29]:
# Read with "::" as separator
df = pd.read_csv("users.dat", sep="::", engine="python", 
                 names=["UserID", "Gender", "Age", "Occupation", "Zipcode"])


print(df.head())



   UserID Gender  Age  Occupation Zipcode
0       1      F    1          10   48067
1       2      M   56          16   70072
2       3      M   25          15   55117
3       4      M   45           7   02460
4       5      M   25          20   55455


In [19]:
df1 = pd.read_csv("ratings.dat", 
                      sep="::", 
                      engine="python", 
                      names=["user_id", "content_id", "rating", "timestamp"])

print(ratings.head())


   user_id  content_id  rating  timestamp
0        1        1193       5  978300760
1        1         661       3  978302109
2        1         914       3  978301968
3        1        3408       4  978300275
4        1        2355       5  978824291


In [27]:
movies = pd.read_csv(
    "movies.dat",
    sep="::",
    engine="python",
    encoding="latin-1",   # <-- Fix UnicodeDecodeError
    names=["content_id", "title", "genres"]
)

print(movies.head())

   content_id                               title  \
0           1                    Toy Story (1995)   
1           2                      Jumanji (1995)   
2           3             Grumpier Old Men (1995)   
3           4            Waiting to Exhale (1995)   
4           5  Father of the Bride Part II (1995)   

                         genres  
0   Animation|Children's|Comedy  
1  Adventure|Children's|Fantasy  
2                Comedy|Romance  
3                  Comedy|Drama  
4                        Comedy  


In [31]:
pip install requests


Note: you may need to restart the kernel to use updated packages.


In [45]:
import pandas as pd
import requests
import re
import time

API_KEY = "8f1d1b01ccbc887118a61d69923a7af7"

# Load movies.dat
movies = pd.read_csv(
    "movies.dat",
    sep="::",
    engine="python",
    encoding="latin-1",
    names=["content_id", "title", "genres"]
)

def extract_title_year(title):
    """Extracts movie title and year from 'Title (Year)' format"""
    match = re.match(r"^(.*)\((\d{4})\)$", title.strip())
    if match:
        return match.group(1).strip(), match.group(2)
    return title, None

def get_movie_description(title):
    """Fetch movie description from TMDB API given a title"""
    movie_title, year = extract_title_year(title)
    url = f"https://api.themoviedb.org/3/search/movie"
    params = {"api_key": API_KEY, "query": movie_title}
    if year:
        params["year"] = year

    for attempt in range(3):  # retry 3 times if error
        try:
            response = requests.get(url, params=params, timeout=10)
            if response.status_code == 200:
                data = response.json()
                if data["results"]:
                    return data["results"][0]["overview"]  # description
                return None
            else:
                print(f"⚠️ Failed for {title}: {response.status_code}")
        except requests.exceptions.RequestException as e:
            print(f"⚠️ Error fetching {title}: {e} (attempt {attempt+1})")
            time.sleep(2)
    return None

# Example: Fetch descriptions for first 5 movies only (test)
movies["description"] = movies["title"].head(5).apply(lambda x: get_movie_description(x))

print(movies.head(10))

# Save results to CSV
movies.to_csv("movies_with_descriptions.csv", index=False, encoding="utf-8")


   content_id                               title  \
0           1                    Toy Story (1995)   
1           2                      Jumanji (1995)   
2           3             Grumpier Old Men (1995)   
3           4            Waiting to Exhale (1995)   
4           5  Father of the Bride Part II (1995)   
5           6                         Heat (1995)   
6           7                      Sabrina (1995)   
7           8                 Tom and Huck (1995)   
8           9                 Sudden Death (1995)   
9          10                    GoldenEye (1995)   

                         genres  \
0   Animation|Children's|Comedy   
1  Adventure|Children's|Fantasy   
2                Comedy|Romance   
3                  Comedy|Drama   
4                        Comedy   
5         Action|Crime|Thriller   
6                Comedy|Romance   
7          Adventure|Children's   
8                        Action   
9     Action|Adventure|Thriller   

                              

In [53]:
dfr.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3883 entries, 0 to 3882
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   content_id   3883 non-null   int64 
 1   title        3883 non-null   object
 2   genres       3883 non-null   object
 3   description  5 non-null      object
dtypes: int64(1), object(3)
memory usage: 121.5+ KB


In [56]:
dfr = pd.read_csv("movies_with_descriptions.csv")
dfr.head()

,content_id,title,genres,description
0,1,Toy Story (1995),Animation|Children's|Comedy,NaN
1,2,Jumanji (1995),Adventure|Children's|Fantasy,NaN
2,3,Grumpier Old Men (1995),Comedy|Romance,NaN
3,4,Waiting to Exhale (1995),Comedy|Drama,NaN
4,5,Father of the Bride Part II (1995),Comedy,NaN


In [59]:
movies.head()

,content_id,title,genres,description
0,1,Toy Story (1995),Animation|Children's|Comedy,None
1,2,Jumanji (1995),Adventure|Children's|Fantasy,None
2,3,Grumpier Old Men (1995),Comedy|Romance,None
3,4,Waiting to Exhale (1995),Comedy|Drama,None
4,5,Father of the Bride Part II (1995),Comedy,None


In [67]:
# Read movies.dat file
movies = pd.read_csv(
    "movies.dat",
    sep="::",
    engine="python",   # because "::" is multi-character separator
    encoding="latin-1",
    names=["content_id", "title", "genres"]
)

# Save to CSV
movies.to_csv("movies.csv", index=False, encoding="utf-8")

print("✅ movies.dat converted to movies.csv")
print(movies.head())


✅ movies.dat converted to movies.csv
   content_id                               title  \
0           1                    Toy Story (1995)   
1           2                      Jumanji (1995)   
2           3             Grumpier Old Men (1995)   
3           4            Waiting to Exhale (1995)   
4           5  Father of the Bride Part II (1995)   

                         genres  
0   Animation|Children's|Comedy  
1  Adventure|Children's|Fantasy  
2                Comedy|Romance  
3                  Comedy|Drama  
4                        Comedy  


In [71]:
import pandas as pd
import requests
import re
import time

API_KEY = "8f1d1b01ccbc887118a61d69923a7af7"

# Load movies.csv
movies = pd.read_csv("movies.csv")

def clean_title(title):
    """Remove year in parentheses from title"""
    return re.sub(r"\(\d{4}\)", "", title).strip()

def get_movie_description(title):
    """Fetch movie description from TMDb API given a title"""
    url = f"https://api.themoviedb.org/3/search/movie?api_key={API_KEY}&query={title}"
    try:
        response = requests.get(url)
        if response.status_code == 200:
            data = response.json()
            if data["results"]:
                return data["results"][0]["overview"]  # take first match
    except Exception as e:
        print(f"Error for {title}: {e}")
    return None

# Clean titles
movies["clean_title"] = movies["title"].apply(clean_title)

# Fetch descriptions
descriptions = []
for i, title in enumerate(movies["clean_title"]):
    desc = get_movie_description(title)
    descriptions.append(desc)
    print(f"{i+1}/{len(movies)} Processed: {title} -> {desc}")
    time.sleep(0.25)  # avoid API rate limit

movies["description"] = descriptions

# Save to new CSV
movies.to_csv("movies_with_descriptions.csv", index=False, encoding="utf-8")

print("✅ Movies with descriptions saved to movies_with_descriptions.csv")


1/3883 Processed: Toy Story -> Led by Woody, Andy's toys live happily in his room until Andy's birthday brings Buzz Lightyear onto the scene. Afraid of losing his place in Andy's heart, Woody plots against Buzz. But when circumstances separate Buzz and Woody from their owner, the duo eventually learns to put aside their differences.
2/3883 Processed: Jumanji -> When siblings Judy and Peter discover an enchanted board game that opens the door to a magical world, they unwittingly invite Alan -- an adult who's been trapped inside the game for 26 years -- into their living room. Alan's only hope for freedom is to finish the game, which proves risky as all three find themselves running from giant rhinoceroses, evil monkeys and other terrifying creatures.
3/3883 Processed: Grumpier Old Men -> A family wedding reignites the ancient feud between next-door neighbors and fishing buddies John and Max. Meanwhile, a sultry Italian divorcée opens a restaurant at the local bait shop, alarming the loc

In [ ]:
import pandas as pd
import requests
import re
import time

API_KEY = "8f1d1b01ccbc887118a61d69923a7af7"

# Load movies.csv
movies = pd.read_csv("movies.csv")

def clean_title(title):
    """Remove year in parentheses from title"""
    return re.sub(r"\(\d{4}\)", "", title).strip()

def get_movie_info(title):
    """Fetch movie description + director from TMDb"""
    search_url = f"https://api.themoviedb.org/3/search/movie?api_key={API_KEY}&query={title}"
    try:
        search_resp = requests.get(search_url)
        if search_resp.status_code == 200:
            data = search_resp.json()
            if data["results"]:
                movie_id = data["results"][0]["id"]
                description = data["results"][0].get("overview")

                # now fetch credits for director
                credits_url = f"https://api.themoviedb.org/3/movie/{movie_id}/credits?api_key={API_KEY}"
                credits_resp = requests.get(credits_url)
                director = None
                if credits_resp.status_code == 200:
                    crew = credits_resp.json().get("crew", [])
                    for member in crew:
                        if member["job"] == "Director":
                            director = member["name"]
                            break

                return description, director
    except Exception as e:
        print(f"Error for {title}: {e}")
    return None, None

# Clean titles
movies["clean_title"] = movies["title"].apply(clean_title)

# Fetch descriptions + directors
descriptions = []
directors = []
for i, title in enumerate(movies["clean_title"]):
    desc, director = get_movie_info(title)
    descriptions.append(desc)
    directors.append(director)
    print(f"{i+1}/{len(movies)} Processed: {title} -> Director: {director}")
    time.sleep(0.25)  # stay under TMDb rate limits

movies["description"] = descriptions
movies["director"] = directors

# Save new CSV
movies.to_csv("movies_with_desc_directors.csv", index=False, encoding="utf-8")

print("✅ Movies with descriptions + directors saved to movies_with_desc_directors.csv")


1/3883 Processed: Toy Story -> Director: John Lasseter
2/3883 Processed: Jumanji -> Director: Joe Johnston
3/3883 Processed: Grumpier Old Men -> Director: Howard Deutch
4/3883 Processed: Waiting to Exhale -> Director: Forest Whitaker
5/3883 Processed: Father of the Bride Part II -> Director: Charles Shyer
6/3883 Processed: Heat -> Director: Michael Mann
7/3883 Processed: Sabrina -> Director: Rocky Soraya
8/3883 Processed: Tom and Huck -> Director: Peter Hewitt
9/3883 Processed: Sudden Death -> Director: Harley Longstaff
10/3883 Processed: GoldenEye -> Director: Martin Campbell
11/3883 Processed: American President, The -> Director: Rob Reiner
12/3883 Processed: Dracula: Dead and Loving It -> Director: Mel Brooks
13/3883 Processed: Balto -> Director: Simon Wells
14/3883 Processed: Nixon -> Director: Oliver Stone
15/3883 Processed: Cutthroat Island -> Director: Renny Harlin
16/3883 Processed: Casino -> Director: Martin Scorsese
17/3883 Processed: Sense and Sensibility -> Director: Ang Le

In [7]:
import pandas as pd
import requests
import re
import time
import os

API_KEY = "8f1d1b01ccbc887118a61d69923a7af7"

# Load CSV (if enriched file exists, resume from there)
if os.path.exists("movies_enriched.csv"):
    movies = pd.read_csv("movies_enriched.csv")
    print("✅ Resuming from movies_enriched.csv")
else:
    movies = pd.read_csv("movies.csv")

# --- Step 1: Split title + year ---
def split_title_year(title):
    match = re.match(r"^(.*)\((\d{4})\)$", title.strip())
    if match:
        return match.group(1).strip(), int(match.group(2))
    return title, None

if "title_without_year" not in movies.columns or "year" not in movies.columns:
    movies[["title_without_year", "year"]] = movies["title"].apply(
        lambda x: pd.Series(split_title_year(x))
    )

# --- Step 2: Safe GET with retries ---
def safe_get(url, retries=5, delay=2):
    for attempt in range(retries):
        try:
            resp = requests.get(url, timeout=15)
            if resp.status_code == 200:
                return resp
            elif resp.status_code == 429:  # Rate limit
                wait_time = int(resp.headers.get("Retry-After", 10))
                print(f"⚠️ Rate limit hit! Waiting {wait_time} seconds...")
                time.sleep(wait_time)
            else:
                print(f"⚠️ Unexpected status {resp.status_code} for {url}")
        except requests.exceptions.RequestException as e:
            print(f"⚠️ Attempt {attempt+1} failed: {e}")
            time.sleep(delay * (attempt + 1))  # exponential backoff
    return None

# --- Step 3: Fetch description + director + cast ---
def get_movie_info(title, year=None, retries=3):
    for attempt in range(retries):
        desc, director, cast = None, None, None

        # Search movie
        search_url = f"https://api.themoviedb.org/3/search/movie?api_key={API_KEY}&query={title}"
        if year:
            search_url += f"&year={year}"

        search_resp = safe_get(search_url)
        if not search_resp:
            print(f"❌ Search failed for {title}, retrying...")
            time.sleep(2 * (attempt + 1))
            continue

        data = search_resp.json()
        if not data["results"]:
            return None, None, None

        movie_id = data["results"][0]["id"]
        desc = data["results"][0].get("overview")

        # Get credits
        credits_url = f"https://api.themoviedb.org/3/movie/{movie_id}/credits?api_key={API_KEY}"
        credits_resp = safe_get(credits_url)
        if credits_resp:
            credits = credits_resp.json()
            crew = credits.get("crew", [])
            cast_list = credits.get("cast", [])

            # Directors
            directors = [m["name"] for m in crew if m.get("job") == "Director"]
            director = ", ".join(directors) if directors else None

            # Top 5 cast
            cast = ", ".join([c["name"] for c in cast_list[:5]]) if cast_list else None

        return desc, director, cast

    # If all retries failed
    print(f"❌ Giving up on {title}")
    return None, None, None

# --- Step 4: Add empty columns if missing ---
for col in ["description", "director", "cast"]:
    if col not in movies.columns:
        movies[col] = None

# --- Step 5: Process all movies ---
for i, row in movies.iterrows():
    # Skip if already filled
    if pd.notna(row["description"]) and pd.notna(row["director"]) and pd.notna(row["cast"]):
        continue  

    desc, director, cast = get_movie_info(row["title_without_year"], row["year"])
    movies.at[i, "description"] = desc
    movies.at[i, "director"] = director
    movies.at[i, "cast"] = cast

    print(f"{i+1}/{len(movies)} ✅ {row['title']} -> Director: {director}")

    # Save progress every 100 movies
    if (i + 1) % 100 == 0:
        movies.to_csv("movies_enriched.csv", index=False, encoding="utf-8")
        print("💾 Progress saved at", i+1)

    time.sleep(1.5)  # respect TMDb rate limits

# --- Step 6: Final save ---
movies.to_csv("movies_enriched.csv", index=False, encoding="utf-8")
print("🎉 All movies enriched and saved to movies_enriched.csv")


1/3883 ✅ Toy Story (1995) -> Director: John Lasseter
2/3883 ✅ Jumanji (1995) -> Director: Joe Johnston
3/3883 ✅ Grumpier Old Men (1995) -> Director: Howard Deutch
4/3883 ✅ Waiting to Exhale (1995) -> Director: Forest Whitaker
5/3883 ✅ Father of the Bride Part II (1995) -> Director: Charles Shyer
6/3883 ✅ Heat (1995) -> Director: Michael Mann
7/3883 ✅ Sabrina (1995) -> Director: Sydney Pollack
8/3883 ✅ Tom and Huck (1995) -> Director: Peter Hewitt
9/3883 ✅ Sudden Death (1995) -> Director: Peter Hyams
10/3883 ✅ GoldenEye (1995) -> Director: Martin Campbell
11/3883 ✅ American President, The (1995) -> Director: Rob Reiner
12/3883 ✅ Dracula: Dead and Loving It (1995) -> Director: Mel Brooks
13/3883 ✅ Balto (1995) -> Director: Simon Wells
14/3883 ✅ Nixon (1995) -> Director: Oliver Stone
15/3883 ✅ Cutthroat Island (1995) -> Director: Renny Harlin
16/3883 ✅ Casino (1995) -> Director: Martin Scorsese
17/3883 ✅ Sense and Sensibility (1995) -> Director: Ang Lee
18/3883 ✅ Four Rooms (1995) -> Dire